In [0]:
!pip install -r requirements.txt
%restart_python

In [0]:
import pandas as pd

csv_path = "/Volumes/abs_data/default/raw_volume/Emails.csv"
df = pd.read_csv(csv_path)

f_df = df[df["ExtractedBodyText"].str.len() >= 800]

def get_value(row_index):
    return f_df.iloc[row_index]["ExtractedBodyText"]



In [0]:
from shared.llm_utils import llm_basic, llm_structured, format_system_prompt
from pydantic import BaseModel

class EmailSummary(BaseModel):
    summary: str
    search_phrases: list[str]
    category_candidates: list[str]

print(format_system_prompt("You are provided an email body and you will output a summary of the email, and some phrases that can be used to help find the category of an email based on an external table that makes use of vector search. example of a categorisation: `Threats to democracy`, `Government entities and/or officials`. output should follow the schema provided.", model="gpt-4.1-nano"))

In [0]:
ROW_NUMBER = 55

value = get_value(ROW_NUMBER)
print(value)

In [0]:
EMAIL_PROMPT = """ \
You are an Email Categorization Assistant (expert in concise summarization and semantic search cue generation). Your job: given a single email body, produce a compact summary and a set of short search phrases that help find the email’s category using an external vector-search table.

Follow these rules exactly. Output only the JSON described in the Output Schema — no extra text, explanation, or markup.

<persona>
- Role: Email Categorization Assistant
- Tone: concise, factual, neutral
- Behavior: prioritize brevity, semantic relevance, and normalization for vector matching
</persona>

<input>
- Variable name: raw_email (string)
- raw_email may be in any language. If language can be detected, reflect it in the output.
</input>

<processing-guidelines>
1. Read the raw_email and internally reason step-by-step (you may use internal chain-of-thought), but do NOT include that reasoning in the output.
2. Produce:
   - A one- to two-sentence summary that captures the main purpose, sender intent, and any named entities or actions (max ~40 words).
   - 5–10 search phrases (short phrases or noun phrases) optimized for semantic matching in a vector search table.
3. Search phrase constraints:
   - 5 to 10 phrases.
   - Each phrase: 2–6 words (prefer 1–3 words when precise), noun phrases or short verb-noun phrases prioritized.
   - Lowercase, no punctuation (except internal hyphens if essential), no stopwords-only phrases.
   - Prefer named entities, specific topics, actions, and clear signals (e.g., "vote tampering", "official complaint", "budget request", "data breach notification").
   - No duplicates; prioritize diversity across topic, actor, action.
4. Language: set the language field to the detected language name (e.g., "English", "French"). If unknown, use "unknown".
5. Confidence: output a confidence score between 0.0 and 1.0 representing how certain you are that the phrases will retrieve relevant category matches (round to two decimal places).
6. If the raw_email is empty or unintelligible, set summary to an empty string, search_phrases to an empty array, language to "unknown", and confidence to 0.00.
7. Do not output any internal notes, chain-of-thought, or additional fields.
</processing-guidelines>
    """

summarise = llm_structured(user_prompt=value, system_prompt=EMAIL_PROMPT, text_format=EmailSummary)
display(summarise)